# Basic Pandas + EDA (Studi Kasus Titanic)

Notebook ini disusun untuk mahasiswa agar memahami alur analisis data yang runtut: baca data, cek kualitas, eksplorasi, visualisasi, insight, dan penanganan missing value.

## Target belajar
1. Membaca dan memahami struktur dataset
2. Mendeteksi missing value (angka dan visual)
3. Melakukan EDA dasar dan visualisasi insight
4. Menangani missing value dengan strategi sederhana
5. Menyusun insight dari data

## 1) Install dan import library
Jika environment belum lengkap, jalankan cell install di bawah ini satu kali.

In [ ]:
# Uncomment jika perlu
# !pip install -q pandas matplotlib seaborn missingno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

## 2) Membaca dataset Titanic
Dataset: 08-Text Mining/titanic.csv

In [ ]:
path_data = "titanic.csv"
df = pd.read_csv(path_data)
print("Ukuran data:", df.shape)
df.head()

## 3) Memahami struktur data
Cek nama kolom, tipe data, dan contoh isi data.

In [ ]:
print(df.columns.tolist())
print()
df.info()

## 4) Statistik deskriptif awal
Gunakan describe untuk melihat ringkasan nilai numerik dan kategori.

In [ ]:
df.describe(include="all")

## 5) Cek missing value (angka)
Kita hitung jumlah dan persentase missing di tiap kolom.

In [ ]:
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)
missing_table

## 6) Visualisasi missing value dengan missingno
Visual ini membantu mahasiswa melihat pola missing secara cepat.

In [ ]:
msno.matrix(df)
plt.show()

msno.bar(df)
plt.show()

## 7) EDA univariat: distribusi umur dan tarif
Kita lihat sebaran umur penumpang dan tarif tiket.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Age"], kde=True, bins=10, ax=axes[0], color="steelblue")
axes[0].set_title("Distribusi Umur")

sns.boxplot(x=df["Fare"], ax=axes[1], color="darkorange")
axes[1].set_title("Boxplot Fare")

plt.tight_layout()
plt.show()

## 8) EDA kategorikal: komposisi kelas dan embarkasi
Insight awal: penumpang paling banyak dari kelas berapa dan pelabuhan mana.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="Pclass", ax=axes[0], palette="Blues")
axes[0].set_title("Jumlah Penumpang per Kelas")

sns.countplot(data=df, x="Embarked", ax=axes[1], palette="Greens")
axes[1].set_title("Jumlah Penumpang per Embarked")

plt.tight_layout()
plt.show()

## 9) EDA bivariat: faktor yang berkaitan dengan survival
Kita bandingkan survival rate berdasarkan jenis kelamin dan kelas.

In [ ]:
survival_by_sex = df.groupby("Sex")["Survived"].mean().reset_index()
survival_by_pclass = df.groupby("Pclass")["Survived"].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=survival_by_sex, x="Sex", y="Survived", ax=axes[0], palette="Set2")
axes[0].set_title("Survival Rate per Gender")
axes[0].set_ylim(0, 1)

sns.barplot(data=survival_by_pclass, x="Pclass", y="Survived", ax=axes[1], palette="Set1")
axes[1].set_title("Survival Rate per Pclass")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("Survival by Sex:")
print(survival_by_sex)
print("\nSurvival by Pclass:")
print(survival_by_pclass)

## 10) Segmentasi sederhana: kelompok umur
Kelompok umur membantu insight yang lebih mudah dibaca.

In [ ]:
df_age = df.copy()
bins = [0, 12, 18, 35, 60, 100]
labels = ["Anak", "Remaja", "Dewasa Muda", "Dewasa", "Lansia"]
df_age["AgeGroup"] = pd.cut(df_age["Age"], bins=bins, labels=labels)

age_survival = df_age.groupby("AgeGroup", observed=False)["Survived"].mean().reset_index()
age_survival

In [ ]:
sns.barplot(data=age_survival, x="AgeGroup", y="Survived", palette="coolwarm")
plt.title("Survival Rate per Kelompok Umur")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

## 11) Menangani missing value
Strategi basic yang mudah dijelaskan:
1. Age diisi median per Pclass
2. Fare diisi median global
3. Embarked diisi modus
4. Cabin dibiarkan atau diberi label Unknown

In [ ]:
df_clean = df.copy()

df_clean["Age"] = df_clean.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))
df_clean["Fare"] = df_clean["Fare"].fillna(df_clean["Fare"].median())
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
df_clean["Cabin"] = df_clean["Cabin"].fillna("Unknown")

print("Missing sebelum:")
print(df.isna().sum())
print("\nMissing sesudah:")
print(df_clean.isna().sum())

## 12) Validasi hasil cleaning
Bandingkan statistik ringkas sebelum dan sesudah cleaning.

In [ ]:
print("Sebelum cleaning:")
print(df[["Age", "Fare"]].describe())
print("\nSesudah cleaning:")
print(df_clean[["Age", "Fare"]].describe())

## 13) Insight ringkas (contoh)
1. Survival rate perempuan cenderung lebih tinggi.
2. Penumpang kelas 1 cenderung survival lebih tinggi dibanding kelas 3.
3. Missing value terbesar ada pada kolom Cabin, sehingga perlu strategi khusus.

## Tugas mahasiswa
1. Buat visual survival rate berdasarkan Embarked.
2. Cari 5 penumpang dengan Fare tertinggi.
3. Coba skenario cleaning lain: drop kolom Cabin, lalu bandingkan insight.